# 13 — JSON, CSV, and Data Pipeline Design

In [ ]:
# Setup: run this first. On Google Colab (or anywhere the course files are missing) it
# downloads the course and moves into this notebook's folder. Elsewhere it does nothing.
import os, sys
if not os.path.exists("../README.md"):
    course = os.path.join(os.path.expanduser("~"), "python-notebook")
    if not os.path.isdir(course):
        os.system(f"git clone -q --depth 1 https://github.com/SwarmLadder/python-notebook {course}")
    os.chdir(os.path.join(course, "intermediate", "notebooks"))
    sys.path.insert(0, os.getcwd())

In [ ]:
import json

from lesson import Exercises, ___, run

check = Exercises('intermediate/13', {1: 'e0d61eb8bbc16728', 2: '21ca10da0018ff55', 3: 'f195fc8edac0baea', 4: 'ce4c7ca8b634f805', 5: 'ab1670767dc0bdad', 6: '12b13cdbc5fd55a0', 7: 'ed5733434771f350', 8: 'b9c8a34b0ba75bd5', 9: '2411fe8d4520be7b'}, practice='intermediate_03_data_quality_and_testing', independent='summarize_sales')

## 1. JSON

- `json.loads(text)` turns JSON text into Python; `json.dumps(value)` goes the other way.
- JSON objects become dicts, arrays lists, `true/false/null` become `True/False/None`.
- Bad JSON raises `json.JSONDecodeError` (a `ValueError`). Valid JSON can still have the wrong shape: check it.

In [ ]:
import json
data = json.loads('{"products": [{"sku": "A1", "price": 9.5}], "ok": true, "next": null}')
print(data["products"][0]["price"], data["ok"], data["next"])
print(json.dumps({"a": [1, 2]}, sort_keys=True))
try:
    json.loads("{oops}")
except json.JSONDecodeError as error:
    print("invalid:", error.msg)

**Exercise 1.** Parse the JSON and get the second item's name.

In [ ]:
import json
data = json.loads('{"items": [{"name": "pen"}, {"name": "ink"}]}')
name = data["items"][___]["name"]
check(1, name)

**Exercise 2.** What do JSON `true` and `null` become? Predict the tuple.

In [ ]:
import json
values = tuple(json.loads("[true, null]"))
predicted = ___
check(2, predicted)

**Exercise 3.** Check the shape: is `products` present and a list?

In [ ]:
import json
def has_products(text):
    data = json.loads(text)
    return isinstance(data, dict) and isinstance(data.get("products"), ___)

check(3, [has_products(t) for t in ['{"products": []}', '{"products": 3}', "[1]"]])

### 1. `parse_products`

Parse `{"products": [...]}`; raise `ValueError` for bad JSON or shape.

In [ ]:
def parse_products(json_text: str) -> list[dict]:
    """Parse '{"products": [...]}' and return the list.

    Raise ValueError if the JSON is invalid, the top level is not an object, or
    'products' is missing or not a list.
    """
    try:
        data = json.loads(json_text)
    except json.JSONDecodeError as error:
        raise ValueError(f"invalid JSON: {error}") from error
    if ___:
        raise ValueError("expected an object with a 'products' list")
    return ___

In [ ]:
check('parse_products')

## 2. CSV

- `csv.DictReader(io.StringIO(text))` gives one dict per row, keyed by the header.
- Every CSV value is a string: convert with `float()`/`int()` and catch `ValueError` for bad rows.
- `enumerate(reader, start=2)` gives line numbers when line 1 is the header.

In [ ]:
import csv, io
text = "name,price\npen,1.5\nink,oops\npad,3"
rows, rejected = [], []
for line_number, row in enumerate(csv.DictReader(io.StringIO(text)), start=2):
    try:
        rows.append({"name": row["name"], "price": float(row["price"])})
    except ValueError:
        rejected.append(line_number)
print(rows, rejected)

**Exercise 4.** What type is every value from `DictReader`? Write its name.

In [ ]:
import csv, io
row = next(csv.DictReader(io.StringIO("qty\n3")))
predicted = ___
check(4, predicted)

**Exercise 5.** Read the rows and total the `qty` column as ints.

In [ ]:
import csv, io
text = "item,qty\npen,3\nink,4"
total = sum(___ for row in csv.DictReader(io.StringIO(text)))
check(5, total)

**Exercise 6.** With the header as line 1, which line numbers are rejected?

In [ ]:
import csv, io
text = "name,price\na,1\nb,x\nc,2\nd,?"
rejected = []
for line_number, row in enumerate(csv.DictReader(io.StringIO(text)), start=___):
    try:
        float(row["price"])
    except ValueError:
        rejected.append(line_number)
check(6, rejected)

### 2. `clean_csv`

Valid rows with float prices, plus the rejected line numbers.

In [ ]:
def clean_csv(text: str) -> tuple[list[dict], list[int]]:
    """Read CSV text with columns name,price. Return (valid rows, rejected line numbers).

    Valid rows have price converted to float. Line numbers count the header as line 1.
    """
    raise NotImplementedError

In [ ]:
check('clean_csv')

## 3. Config lines and pipelines

- Parse `key = value` lines with `line.split("=", 1)` and `.strip()` both sides.
- Skip blank lines and `#` comments; raise `ValueError` naming the line number for anything else that's wrong.
- Build pipelines from small steps (parse → validate → aggregate) so each can be tested alone.

In [ ]:
text = "# app\nhost = example.com\n\nport = 8080"
settings = {}
for number, line in enumerate(text.splitlines(), start=1):
    stripped = line.strip()
    if not stripped or stripped.startswith("#"):
        continue
    key, value = stripped.split("=", 1)
    settings[key.strip()] = value.strip()
print(settings)

**Exercise 7.** Split on the first `=` only: what does `"url = a=b".split("=", 1)` give?

In [ ]:
predicted = ___
check(7, predicted)

**Exercise 8.** Which lines should be skipped? Finish the condition.

In [ ]:
lines = ["# comment", "", "a = 1", "   "]
kept = [l for l in lines if not (not l.strip() or l.strip().startswith("" + ___))]
check(8, kept)

**Exercise 9.** Total sales per region, skipping amounts that aren't numbers.

In [ ]:
rows = [{"region": "N", "amount": "10.5"}, {"region": "S", "amount": "x"}, {"region": "N", "amount": "2"}]
totals = {}
for row in rows:
    try:
        amount = float(row["amount"])
    except ValueError:
        continue
    totals[row["region"]] = totals.get(row["region"], ___) + amount
check(9, totals)

## Challenges

Real functions, graded by tests. Complete each implementation and run its check. The independent build supplies only the interface.

### 3. `parse_settings`

Parse `key = value` lines, skipping blanks and comments.

In [ ]:
def parse_settings(text: str) -> dict[str, str]:
    """Parse 'key = value' lines. Skip blank lines and lines starting with '#'.

    Raise ValueError mentioning the line number (e.g. 'line 3') for a line without '='.
    """
    raise NotImplementedError

In [ ]:
check('parse_settings')

### Independent build — `summarize_sales`

Each row has a region. Sum numeric amounts (possibly given as text) by region and round totals to two decimals. Skip missing or nonnumeric amounts. Regions with no valid rows are absent.

Implement the interface below, run its check, and explain one boundary case with a test of your own.

**Delayed revision.** Finance received one row with a missing amount and another with the text "0". Write a test that distinguishes an absent total from a recorded zero.

In [ ]:
def summarize_sales(rows: list[dict]) -> dict[str, float]:
    raise NotImplementedError

In [ ]:
check('summarize_sales')

## Quick quiz

Replace each `'?'` in the cell below with your choice and run it. The answers are hidden, but each one is checked.

**1.** Which converts a JSON string into Python objects?

   - **a)** eval
   - **b)** json.loads
   - **c)** json.dumps

**2.** Why is eval() unsafe for parsing data?

   - **a)** It doesn't handle numbers
   - **b)** It can execute arbitrary code
   - **c)** It's slow

**3.** A good partial-failure policy for bad CSV rows is…

   - **a)** Collect and report rejected rows with their line numbers
   - **b)** Crash on the first bad row, always
   - **c)** Ignore them silently

In [ ]:
from quiz import Quiz

quiz = Quiz('intermediate/13', {1: '6bceaacdd613', 2: 'af221cbe4dc0', 3: '8bb1e89db1e9'})
quiz.answer(1, '?')
quiz.answer(2, '?')
quiz.answer(3, '?')

**Next:** [14 — Unit Testing, Fixtures, and Mocking](14_unittest_and_mocking.ipynb)